In [1]:
import os 
from collections import Counter

for p in ["p01","p03","p05"]:
    dossier = f"../data/Test_technique/{p}/food-images/"
    if not os.path.exists(dossier):
        print(p,"pas de dossier food-images")
        continue
    fichiers=os.listdir(dossier)
    extensions=Counter(os.path.splitext(f)[1].lower() for f in fichiers)
    print(p,len(fichiers),"fichiers",dict(extensions))
    print("examples:",fichiers[:3])
    

p01 321 fichiers {'.jpeg': 320, '.png': 1}
examples: ['IMG_9078.jpeg', 'IMG_9351.jpeg', 'IMG_9214.jpeg']
p03 136 fichiers {'.jpg': 136}
examples: ['IMG_2603.jpg', 'IMG_2617.jpg', 'IMG_2602.jpg']
p05 186 fichiers {'.jpg': 186}
examples: ['IMG_2603.jpg', 'IMG_2617.jpg', 'IMG_2398.jpg']


In [2]:
from PIL import Image
from PIL.ExifTags import TAGS

chemin = "../data/Test_technique/p01/food-images/IMG_9078.jpeg"
img = Image.open(chemin)
exif = img.getexif()

for tag_id, valeur in exif.items():
    print(TAGS.get(tag_id, tag_id), ":", valeur)

print("---")
details = exif.get_ifd(0x8769)
for tag_id, valeur in details.items():
    print(TAGS.get(tag_id, tag_id), ":", valeur)

GPSInfo : 2040
ResolutionUnit : 2
ExifOffset : 206
Make : Apple
Model : iPhone XS
Software : 13.3.1
Orientation : 6
DateTime : 2020:02:18 13:25:08
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0
---
ExifVersion : b'0231'
ComponentsConfiguration : b'\x01\x02\x03\x00'
ShutterSpeedValue : 6.57470398277718
DateTimeOriginal : 2020:02:18 13:25:08
DateTimeDigitized : 2020:02:18 13:25:08
ApertureValue : 1.6959938128383605
BrightnessValue : 4.412218249650241
ExposureBiasValue : 0.0
MeteringMode : 5
Flash : 24
FocalLength : 4.25
ColorSpace : 65535
ExifImageWidth : 4032
DigitalZoomRatio : 1.1550802139037433
FocalLengthIn35mmFilm : 30
SceneCaptureType : 0
OffsetTime : +01:00
OffsetTimeOriginal : +01:00
OffsetTimeDigitized : +01:00
SubsecTimeOriginal : 072
SubjectLocation : (2014, 1512, 2323, 1394)
SubsecTimeDigitized : 072
ExifImageHeight : 3024
SensingMethod : 2
ExposureTime : 0.010526315789473684
FNumber : 1.8
SceneType : b'\x01'
ExposureProgram : 2
ISOSpeedRatings : 64
ExposureMode :

In [3]:
import pandas as pd

def date_photo(chemin):
    try:
        exif = Image.open(chemin).getexif()
        details = exif.get_ifd(0x8769)
        date = details.get(36867) or exif.get(306)  # DateTimeOriginal, sinon DateTime
        return pd.to_datetime(date, format="%Y:%m:%d %H:%M:%S") if date else pd.NaT
    except Exception:
        return pd.NaT

lignes = []
for p in ["p01", "p03", "p05"]:
    dossier = f"../data/Test_technique/{p}/food-images/"
    for f in sorted(os.listdir(dossier)):
        lignes.append({"participant": p, "fichier": f, "datetime": date_photo(dossier + f)})

photos = pd.DataFrame(lignes)
print(photos["datetime"].isna().sum(), "photos sans date")
photos.head()

0 photos sans date


,participant,fichier,datetime
0,p01,IMG_8916.jpeg,2020-02-01 10:03:41
1,p01,IMG_8917.jpeg,2020-02-01 11:39:57
2,p01,IMG_8918.jpeg,2020-02-01 17:03:39
3,p01,IMG_8920.jpeg,2020-02-01 19:16:48
4,p01,IMG_8921.jpeg,2020-02-01 21:04:57


In [4]:
photos["date"] = photos["datetime"].dt.date
photos.groupby("participant")["datetime"].agg(["min", "max", "count"])

,min,max,count
participant,,,
p01,2020-02-01 10:03:41,2020-03-31 23:19:04,321
p03,2020-02-01 10:36:10,2020-03-31 14:47:33,136
p05,2020-02-01 06:55:04,2020-03-29 17:07:31,186


In [5]:
par_jour = photos.groupby(["participant", "date"]).size()
par_jour.groupby("participant").describe()

,count,mean,std,min,25%,50%,75%,max
participant,,,,,,,,
p01,60.0,5.350000,1.811685,1.0,4.0,5.0,7.0,11.0
p03,51.0,2.666667,1.829390,1.0,1.0,2.0,4.0,9.0
p05,47.0,3.957447,3.451260,1.0,1.5,3.0,5.0,14.0


In [6]:
photos["heure"] = photos["datetime"].dt.hour + photos["datetime"].dt.minute / 60

repas_jour = photos.groupby(["participant", "date"]).agg(
    n_photos=("fichier", "size"),
    premier_repas=("heure", "min"),
    dernier_repas=("heure", "max"),
).reset_index()

os.makedirs("../data/processed", exist_ok=True)
photos.drop(columns=["hash"], errors="ignore").to_csv("../data/processed/photos_meta.csv", index=False)
repas_jour.to_csv("../data/processed/repas_jour.csv", index=False)
repas_jour.head()

,participant,date,n_photos,premier_repas,dernier_repas
0,p01,2020-02-01,7,10.050000,23.100000
1,p01,2020-02-02,6,8.166667,22.666667
2,p01,2020-02-03,9,6.666667,23.466667
3,p01,2020-02-04,6,6.416667,21.300000
4,p01,2020-02-05,5,6.116667,20.750000


In [7]:
from transformers import pipeline
from PIL import ImageOps

clf = pipeline("image-classification", model="nateraw/food")
for f in photos[photos["participant"] == "p01"].head(5)["fichier"]:
    img = ImageOps.exif_transpose(Image.open(f"../data/Test_technique/p01/food-images/{f}")).convert("RGB")
    print(f, [(r["label"], round(r["score"], 2)) for r in clf(img, top_k=3)])

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

IMG_8916.jpeg [('peking_duck', 0.6), ('cheese_plate', 0.31), ('foie_gras', 0.02)]
IMG_8917.jpeg [('waffles', 0.17), ('pancakes', 0.15), ('apple_pie', 0.15)]
IMG_8918.jpeg [('edamame', 0.46), ('macarons', 0.11), ('poutine', 0.02)]
IMG_8920.jpeg [('pizza', 0.97), ('omelette', 0.01), ('lasagna', 0.0)]
IMG_8921.jpeg [('frozen_yogurt', 0.15), ('hot_dog', 0.07), ('hot_and_sour_soup', 0.04)]


In [8]:
# test food-101 sur 5 photos de p01 : 1 seule bonne prédiction (pizza)
# pas de boissons dans food-101 et peu de plats du quotidien -> pas utilisable

In [9]:
import pandas as pd

alim = pd.read_xml("../data/ciqual_table/alim_2025_11_03.xml")
const = pd.read_xml("../data/ciqual_table/const_2025_11_03.xml")

print(alim.shape, list(alim.columns))
print(const.shape, list(const.columns))

(3484, 8) ['alim_code', 'alim_nom_fr', 'alim_nom_eng', 'alim_nom_sci', 'alim_grp_code', 'alim_ssgrp_code', 'alim_ssssgrp_code', 'facteur_Jones']
(74, 4) ['const_code', 'const_nom_fr', 'const_nom_eng', 'code_INFOODS']


In [10]:
const[const["const_nom_fr"].str.contains("nergie|Protéines|Glucides|Lipides", case=False, na=False)][["const_code", "const_nom_fr"]]

,const_code,const_nom_fr
0,327,"Energie, Règlement UE N° 1169/2011 (kJ/100 g)"
1,328,"Energie, Règlement UE N° 1169/2011 (kcal/100 g)"
2,332,"Energie, N x facteur Jones, avec fibres (kJ/1..."
3,333,"Energie, N x facteur Jones, avec fibres (kcal..."
19,25000,"Protéines, N x facteur de Jones (g/100 g)"
20,25003,"Protéines, N x 6.25 (g/100 g)"
21,31000,Glucides (g/100 g)
32,40000,Lipides (g/100 g)


In [11]:
compo = pd.read_xml("../data/ciqual_table/compo_2025_11_03.xml")
print(compo.shape, list(compo.columns))
compo.head(10)

(257816, 7) ['alim_code', 'const_code', 'teneur', 'min', 'max', 'code_confiance', 'source_code']


,alim_code,const_code,teneur,min,max,code_confiance,source_code
0,1000,327,1140,NaN,NaN,D,444.0
1,1000,328,274,NaN,NaN,D,444.0
2,1000,332,1140,NaN,NaN,D,444.0
3,1000,333,274,NaN,NaN,D,444.0
4,1000,400,"59,7","58,7","60,4",D,1373.0
5,1000,10000,"0,002",NaN,NaN,D,444.0
6,1000,10004,0,NaN,NaN,C,355.0
7,1000,10110,0,NaN,NaN,C,355.0
8,1000,10120,0,NaN,NaN,D,74.0
9,1000,10150,-,NaN,NaN,NaN,NaN


In [12]:
import numpy as np

codes = {328: "kcal", 25000: "proteines", 31000: "glucides", 40000: "lipides"}
sel = compo[compo["const_code"].isin(codes)].copy()


def nombre(v):
    v = str(v).strip().replace(",", ".")
    if v in ["-", "", "nan"]:
        return np.nan
    if v == "traces" or v.startswith("<"):
        return 0.0
    return float(v)

sel["valeur"] = sel["teneur"].apply(nombre)


nutri = sel.pivot_table(index="alim_code", columns="const_code", values="valeur").rename(columns=codes)
nutri = nutri.join(alim.set_index("alim_code")["alim_nom_fr"])
print(nutri.shape)
nutri.describe()

(3479, 5)


,kcal,proteines,glucides,lipides
count,3341.000000,3455.000000,3414.000000,3464.000000
mean,207.021366,9.114154,17.025818,10.338417
std,172.011400,9.096473,22.639701,15.939776
min,0.000000,0.000000,0.000000,0.000000
25%,72.600000,1.375000,0.930000,0.457500
50%,163.000000,6.350000,6.250000,4.285000
75%,311.000000,15.500000,23.600000,13.700000
max,900.000000,86.900000,99.800000,100.000000


In [13]:

def cherche(mot):
    res = nutri[nutri["alim_nom_fr"].str.contains(mot, case=False, na=False)]
    return res[["alim_nom_fr", "kcal", "proteines", "glucides", "lipides"]]

cherche(r"\bcola\b")

,alim_nom_fr,kcal,proteines,glucides,lipides
alim_code,,,,,
18018,"Cola, sucré",40.100,0.000026,10.00,0.000023
18037,"Cola, sucré, avec édulcorants",24.300,0.000025,6.07,0.000025
18060,"Cola, sans sucres ajoutés, avec édulcorants",1.040,0.000019,0.25,0.000020
18063,"Cola, sans précision sur la teneur en sucres ...",27.500,0.005600,6.88,0.005600
18067,"Cola, sucré, sans caféine",40.900,0.050000,10.20,0.050000
18068,"Cola, sans sucres ajoutés, avec édulcorants, ...",0.058,0.083000,0.13,0.083000


In [14]:
nutri = nutri.join(alim.set_index("alim_code")["alim_nom_eng"])
ciq = nutri.dropna(subset=["kcal", "alim_nom_eng"]).reset_index()
print(len(ciq), "aliments ciqual utilisables")

3341 aliments ciqual utilisables


In [15]:
import torch
from transformers import AutoModel, AutoProcessor

nom_modele = "google/siglip2-base-patch16-224"
model = AutoModel.from_pretrained(nom_modele)
processor = AutoProcessor.from_pretrained(nom_modele)
model.eval()

textes = ["a photo of " + n for n in ciq["alim_nom_eng"]]

vec_txt = []
with torch.no_grad():
    for i in range(0, len(textes), 64):
        inp = processor(text=textes[i:i + 64], padding="max_length", max_length=64,
                        truncation=True, return_tensors="pt")
        out = model.get_text_features(**inp)
        out = out if torch.is_tensor(out) else out.pooler_output
        vec_txt.append(out)

vec_txt = torch.cat(vec_txt)
vec_txt = vec_txt / vec_txt.norm(dim=-1, keepdim=True)
print(vec_txt.shape)

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

torch.Size([3341, 768])


In [16]:
from PIL import Image, ImageOps

def vec_image(chemin):
    img = ImageOps.exif_transpose(Image.open(chemin)).convert("RGB")
    inp = processor(images=img, return_tensors="pt")
    with torch.no_grad():
        v = model.get_image_features(**inp)
    v = v if torch.is_tensor(v) else v.pooler_output
    return v / v.norm(dim=-1, keepdim=True)

exemples = photos[photos["participant"] == "p01"].head(5)
for f in exemples["fichier"]:
    v = vec_image(f"../data/Test_technique/p01/food-images/{f}")
    sim = (v @ vec_txt.T)[0]
    top = sim.topk(5)
    print(f)
    for s, i in zip(top.values, top.indices):
        print("   ", round(s.item(), 3), ciq.loc[i.item(), "alim_nom_eng"])

IMG_8916.jpeg
    0.14  Soft-ripened cheese with bloomy rind (Camembert-type cheese), from cow's milk 
    0.137  Mimolette cheese, half-old, from cow's milk (cured 6 months minimum) 
    0.136  Soft-ripened cheese with bloomy rind, reduced fat (Camembert-type cheese), from cow's milk 
    0.136  Carré de l'Est cheese, from cow's milk 
    0.133  Hard cheese, emmental-type cheese, reduced fat, from cow's milk 
IMG_8917.jpeg
    0.124  Custard dessert, canned (average) 
    0.12  Custard dessert, refrigerated or canned (average) 
    0.119  Milk roll, from bakery 
    0.115  Carbonated beverage or soft drink, without fruit, with sugar(s) and with sweetener(s) 
    0.113  Cola, with sugar(s) and without caffeine 
IMG_8918.jpeg
    0.175  Chocolate-coated peanut 
    0.173  Puffed salty snacks, peanut flavour 
    0.17  Seeds and peanuts, dried, salted (average) 
    0.167  Peanuts coated with savoury biscuit (with salt or flavoured) 
    0.166  Candied peanut 
IMG_8920.jpeg
    0.137  Om

In [17]:
# siglip2 avec toute la table ciqual comme liste d'aliments : 4 photos sur 5 dans la bonne famille, 1 à moitié
# brioche + coca -> viennoiserie et sodas dans le top 5, snacks -> cacahuètes, coca -> sodas, pizza en 2e
# beaucoup mieux que food-101 (1/5), et pas besoin de savoir à l'avance ce qu'il y a sur les photos

In [18]:
grp = pd.read_xml("../data/ciqual_table/alim_grp_2025_11_03.xml")
print(list(grp.columns))
grp.iloc[:, :2].drop_duplicates()

['alim_grp_code', 'alim_grp_nom_fr', 'alim_grp_nom_eng', 'alim_ssgrp_code', 'alim_ssgrp_nom_fr', 'alim_ssgrp_nom_eng', 'alim_ssssgrp_code', 'alim_ssssgrp_nom_fr', 'alim_ssssgrp_nom_eng']


,alim_grp_code,alim_grp_nom_fr
0,1,entrées et plats composés
14,2,"fruits, légumes, légumineuses et oléagineux"
30,3,produits céréaliers
37,4,"viandes, oeufs, poissons"
74,5,produits laitiers
90,6,eaux et autres boissons
102,7,produits sucrés
111,8,glaces et sorbets
115,9,matières grasses
120,10,aides culinaires et ingrédients divers


In [19]:
# boisson = groupe 6 de ciqual
grp_alim = alim.set_index("alim_code")["alim_grp_code"]
ciq["boisson"] = pd.to_numeric(ciq["alim_code"].map(grp_alim), errors="coerce") == 6

cols_n = ["kcal", "proteines", "glucides", "lipides"]
res = []
for n, (p, f) in enumerate(zip(photos["participant"], photos["fichier"])):
    v = vec_image(f"../data/Test_technique/{p}/food-images/{f}")
    top = (v @ vec_txt.T)[0].topk(5).indices.tolist()

    valeurs = ciq.loc[top, cols_n].mean()               # moyenne du top 5, pour 100 g
    portion = 250 if ciq.loc[top[0], "boisson"] else 150
    ligne = (valeurs * portion / 100).to_dict()
    ligne.update({"participant": p, "fichier": f, "aliment": ciq.loc[top[0], "alim_nom_eng"], "portion": portion})
    res.append(ligne)

    if n % 100 == 0:
        print(n, "photos faites")

nutri_photos = pd.DataFrame(res)
nutri_photos.head(10)

0 photos faites
100 photos faites
200 photos faites
300 photos faites
400 photos faites
500 photos faites
600 photos faites


,kcal,proteines,glucides,lipides,participant,fichier,aliment,portion
0,418.800,36.150000,1.1490,30.150000,p01,IMG_8916.jpeg,Soft-ripened cheese with bloomy rind (Camembe...,150
1,213.990,4.691160,31.1640,7.413005,p01,IMG_8917.jpeg,"Custard dessert, canned (average)",150
2,803.400,21.570000,67.5300,47.220000,p01,IMG_8918.jpeg,Chocolate-coated peanut,150
3,282.300,21.783000,10.4940,16.794000,p01,IMG_8920.jpeg,"Omelette with lardoons, homemade",150
4,50.065,0.015467,12.4915,0.004724,p01,IMG_8921.jpeg,"Carbonated beverage or soft drink, without fr...",250
5,445.200,6.273000,86.9640,12.810000,p01,IMG_8922.jpeg,"Chewing gum, sugar level unknown (average)",150
6,749.100,10.815000,87.9600,38.400000,p01,IMG_8923.jpeg,"Puffed salty snacks, bacon flavour",150
7,638.100,8.481000,106.2000,20.289000,p01,IMG_8927.jpeg,"Cereal bar with fruit, fortified with vitamin...",150
8,666.000,10.125000,100.4700,23.880000,p01,IMG_8928.jpeg,Baby biscuit,150
9,301.800,20.433000,19.4250,12.960000,p01,IMG_8929.jpeg,"Soy and wheat burger or bite (not vegan), pre...",150


In [20]:
# portion de 150 g trop grosse pour les snacks (cacahuètes 803 kcal, chewing-gum 445 kcal)
# règle : boisson 250 ml, aliment > 350 kcal/100g -> 40 g, sinon 150 g

In [21]:
# on revient aux valeurs pour 100 g
for c in cols_n:
    nutri_photos[c + "_100g"] = nutri_photos[c] * 100 / nutri_photos["portion"]

# aliments très caloriques (snacks, chocolat, biscuits...) -> petite portion
boisson = nutri_photos["portion"] == 250
dense = nutri_photos["kcal_100g"] > 350
nutri_photos["portion"] = np.where(boisson, 250, np.where(dense, 40, 150))

for c in cols_n:
    nutri_photos[c] = nutri_photos[c + "_100g"] * nutri_photos["portion"] / 100

print(nutri_photos["portion"].value_counts())
nutri_photos[["aliment", "kcal_100g", "portion", "kcal"]].head(10)

portion
150    420
40     118
250    105
Name: count, dtype: int64


,aliment,kcal_100g,portion,kcal
0,Soft-ripened cheese with bloomy rind (Camembe...,279.200,150,418.800
1,"Custard dessert, canned (average)",142.660,150,213.990
2,Chocolate-coated peanut,535.600,40,214.240
3,"Omelette with lardoons, homemade",188.200,150,282.300
4,"Carbonated beverage or soft drink, without fr...",20.026,250,50.065
5,"Chewing gum, sugar level unknown (average)",296.800,150,445.200
6,"Puffed salty snacks, bacon flavour",499.400,40,199.760
7,"Cereal bar with fruit, fortified with vitamin...",425.400,40,170.160
8,Baby biscuit,444.000,40,177.600
9,"Soy and wheat burger or bite (not vegan), pre...",201.200,150,301.800


In [22]:
# après correction : cacahuètes 803 -> 214 kcal, snacks 749 -> 200, barre de céréales 638 -> 170
# 420 photos en plat (150 g), 118 en aliment très calorique (40 g), 105 en boisson (250 ml)
# reste des cas ratés comme le chewing-gum (top 5 mélangé, moyenne sous le seuil) -> limite de la méthode

In [23]:
nutri_photos = nutri_photos.merge(photos[["participant", "fichier", "date"]], on=["participant", "fichier"])

apports = nutri_photos.groupby(["participant", "date"])[cols_n].sum().round(1).reset_index()
apports = apports.rename(columns={"kcal": "kcal_conso", "proteines": "prot_conso",
                                  "glucides": "gluc_conso", "lipides": "lip_conso"})

repas_final = repas_jour.merge(apports, on=["participant", "date"], how="left")
repas_final.to_csv("../data/processed/repas_jour.csv", index=False)

repas_final.groupby("participant")["kcal_conso"].describe()

,count,mean,std,min,25%,50%,75%,max
participant,,,,,,,,
p01,60.0,1350.918333,496.631053,64.6,1027.45,1284.25,1587.35,2712.5
p03,51.0,793.198039,510.569086,80.5,414.80,681.50,1075.25,2078.8
p05,47.0,1081.808511,949.801791,89.7,435.00,839.10,1220.15,3749.1


In [24]:
# kcal consommées estimées par jour : p01 ~1350, p03 ~790, p05 ~1080
# bien en dessous de la réalité (p01 dépense ~3600 kcal/jour d'après la montre)
# portion de 150 g trop petite pour un vrai repas, plusieurs aliments mélangés, repas pas tous photographiés (p03, p05)
# valeur absolue pas fiable, utilisable surtout en relatif (jour vs habitude du participant)

In [25]:
nutri_photos.to_csv("../data/processed/nutri_photos.csv", index=False)